In [1]:
import pysam
from collections import defaultdict
import pandas as pd

In [2]:
def format_to_dataframe(r2_reads, r2_umis):
    # Ensure both dictionaries have the same keys
    all_keys = set(r2_reads.keys()).union(r2_umis.keys())

    # Create a list of rows for the DataFrame
    data = [
        {"location": key, "reads": r2_reads.get(key, 0), "umis": r2_umis.get(key, 0)}
        for key in all_keys
    ]

    # Convert to DataFrame
    df = pd.DataFrame(data)
    df = df.sort_values(by="umis", ascending=False)
    return df

def process_reads_relaxed_cigar(sample_prefix):
    allowed_cigars = {"25S62M", "24S63M", "23S64M"}
    
    distance_list = []
    missing_r1 = 0
    # Define the BAM file path
    bam_path = f"./250225_alignment/{sample_prefix}.sorted.bam"

    # Dictionary to store R1 locations
    r1_locations = {}

    # Dictionaries to store R2 information
    r2_reads = defaultdict(int)  # Total reads per R2 location
    r2_umis = defaultdict(int)   # Unique UMIs (based on R1 locations) per R2 location
    r2_r1s = defaultdict(list)   # List of R1 locations associated with each R2 location

    # First pass: Process R1 reads
    with pysam.AlignmentFile(bam_path, "rb") as bam:
        for read in bam:
            
            # Skip secondary or supplementary alignments
            if read.is_secondary or read.is_supplementary:
                continue
            
            if not read.is_read1:  # Focus on read1 only
                continue
                
            if not read.is_proper_pair:  # Filter out non-properly paired reads
                continue

            # Check quality
            if read.mapping_quality == 60:# and read.cigarstring == "170M":
                # Store R1 location with read name as the key
                r1_locations[read.query_name] = f"{read.reference_name}-{read.reference_start + 1}"

    # Second pass: Process R2 reads
    with pysam.AlignmentFile(bam_path, "rb") as bam:
        for read in bam:
            # Skip secondary or supplementary alignments
            if read.is_secondary or read.is_supplementary:
                continue
            if not read.is_read2:  # Focus on read2 only
                continue
            if not read.is_proper_pair:  # Filter out non-properly paired reads
                continue

            # Check quality and CIGAR
            if read.mapping_quality == 60 and read.cigarstring in allowed_cigars:
                
                if abs(read.template_length) > 1500:
                    continue
                
                # Extract the full read sequence
                seq = read.query_sequence
                
                # Get the length of the softclip
                softclip_len = read.cigartuples[0][1]
                
                # Get how much the softclip deviated from expected softclip length of 25bp
                # i.e. how many bp of the barcode were included in the alignment
                jitter = 25 - softclip_len

                # adjusting the alignment start by how much of the barcode aligned 
                corrected_start = read.reference_start + 1 + jitter
                
                # First 25 bp = soft-clipped region (18 constant + 7 barcode)
                barcode = seq[18:25]

                # Build an R2 "location" key that includes the barcode
                r2_location = f"{read.reference_name}-{corrected_start}-{barcode}"
            
                r2_reads[r2_location] += 1  # Increment total reads for this R2 location

                # Get corresponding R1 location
                read1_loc = r1_locations.get(read.query_name)
                if read1_loc is None:
                    missing_r1 += 1
                    continue  # Skip if no corresponding R1 location is found

                # Check if this R1 location is already associated with this R2 location
                if read1_loc not in r2_r1s[r2_location]:
                    r2_r1s[r2_location].append(read1_loc)  # Add the R1 location to the list
                    r2_umis[r2_location] += 1  # Increment unique UMIs for this R2 location
                    
                    tlen = read.template_length
                    distance = abs(tlen)
                    distance_list.append(distance) 
                    
    result_df = format_to_dataframe(r2_reads, r2_umis)
    print(f"Total R2 reads without corresponding R1: {missing_r1}")
    result_df.to_csv(f"./t7_counts/{sample_prefix}.counts.unique.filtered.relaxed-cigar.tsv", sep='\t', index=False)

    return result_df

In [3]:
print("processing R1")
r1_counts = process_reads_relaxed_cigar("t7_R1_S1")
print("processing R2")
r2_counts = process_reads_relaxed_cigar("t7_R2_S2")
print("processing R3")
r3_counts = process_reads_relaxed_cigar("t7_R3_S3")

print("processing N1")
n1_counts = process_reads_relaxed_cigar("t7_N1_S4")
print("processing N2")
n2_counts = process_reads_relaxed_cigar("t7_N2_S5")
print("processing N3")
n3_counts = process_reads_relaxed_cigar("t7_N3_S6")

processing R1
Total R2 reads without corresponding R1: 6617921
processing R2
Total R2 reads without corresponding R1: 4544443
processing R3
Total R2 reads without corresponding R1: 4311208
processing N1
Total R2 reads without corresponding R1: 20592
processing N2
Total R2 reads without corresponding R1: 17910
processing N3
Total R2 reads without corresponding R1: 16816


In [4]:
def process_reads_loose(sample_prefix):
    distance_list = []
    missing_r1 = 0
    # Define the BAM file path
    bam_path = f"./250225_alignment/{sample_prefix}.sorted.bam"

    # Dictionary to store R1 locations
    r1_locations = {}

    # Dictionaries to store R2 information
    r2_reads = defaultdict(int)  # Total reads per R2 location
    r2_umis = defaultdict(int)   # Unique UMIs (based on R1 locations) per R2 location
    r2_r1s = defaultdict(list)   # List of R1 locations associated with each R2 location

    # First pass: Process R1 reads
    with pysam.AlignmentFile(bam_path, "rb") as bam:
        for read in bam:
            
            # Skip secondary or supplementary alignments
            if read.is_secondary or read.is_supplementary:
                continue
            
            if not read.is_read1:  # Focus on read1 only
                continue
                
            if not read.is_proper_pair:  # Filter out non-properly paired reads
                continue

            # Check quality (loosened to MAPQ >= 30)
            if read.mapping_quality >= 30:
                # Store R1 location with read name as the key
                r1_locations[read.query_name] = f"{read.reference_name}-{read.reference_start + 1}"

    # Second pass: Process R2 reads
    with pysam.AlignmentFile(bam_path, "rb") as bam:
        for read in bam:
            # Skip secondary or supplementary alignments
            if read.is_secondary or read.is_supplementary:
                continue
            if not read.is_read2:  # Focus on read2 only
                continue
            if not read.is_proper_pair:  # Filter out non-properly paired reads
                continue

            # Extract the full read sequence
            seq = read.query_sequence

            # Check quality and that read2 sequence starts with the desired prefix
            if read.mapping_quality < 30:
                continue
            if not seq or not seq.startswith("TAGTTGTGCTTTCCGCGT"):
                continue

            if abs(read.template_length) > 1500:
                continue

            # First 25 bp = 18 constant + 7 barcode
            barcode = seq[18:25]

            # Build an R2 "location" key that includes the barcode
            r2_location = f"{read.reference_name}-{read.reference_start + 1}-{barcode}"
        
            r2_reads[r2_location] += 1  # Increment total reads for this R2 location

            # Get corresponding R1 location
            read1_loc = r1_locations.get(read.query_name)
            if read1_loc is None:
                missing_r1 += 1
                continue  # Skip if no corresponding R1 location is found

            # Check if this R1 location is already associated with this R2 location
            if read1_loc not in r2_r1s[r2_location]:
                r2_r1s[r2_location].append(read1_loc)  # Add the R1 location to the list
                r2_umis[r2_location] += 1  # Increment unique UMIs for this R2 location
                
                tlen = read.template_length
                distance = abs(tlen)
                distance_list.append(distance) 
                    
    result_df = format_to_dataframe(r2_reads, r2_umis)
    print(f"Total R2 reads without corresponding R1: {missing_r1}")
    result_df.to_csv(f"./t7_counts/{sample_prefix}.counts.unique.filtered.loose.tsv", sep='\t', index=False)

    return result_df

In [5]:
print("processing R1")
r1_counts = process_reads_loose("t7_R1_S1")
print("processing R2")
r2_counts = process_reads_loose("t7_R2_S2")
print("processing R3")
r3_counts = process_reads_loose("t7_R3_S3")

processing R1
Total R2 reads without corresponding R1: 311344
processing R2
Total R2 reads without corresponding R1: 195779
processing R3
Total R2 reads without corresponding R1: 188183


In [4]:
print("processing N1")
n1_counts = process_reads_loose("t7_N1_S4")
print("processing N2")
n2_counts = process_reads_loose("t7_N2_S5")
print("processing N3")
n3_counts = process_reads_loose("t7_N3_S6")

processing N1
Total R2 reads without corresponding R1: 1461
processing N2
Total R2 reads without corresponding R1: 1156
processing N3
Total R2 reads without corresponding R1: 1292
